<a href="https://colab.research.google.com/github/wtryab-re/whats-your-eda-basics/blob/main/data_quality_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Installation n Imports

In [112]:
!pip install -q pandas numpy scikit-learn fg-data-profiling matplotlib seaborn

In [113]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_profiling import ProfileReport

In [114]:
pd.set_option("display.max_columns",50)

##Dataset Load and basic evaluation

In [115]:
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [116]:
df.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
202,0,3,male,34.0,0,0,6.4958,S,Third,man,True,NaN,Southampton,no,True
388,0,3,male,NaN,0,0,7.7292,Q,Third,man,True,NaN,Queenstown,no,True
377,0,1,male,27.0,0,2,211.5000,C,First,man,True,C,Cherbourg,no,False
174,0,1,male,56.0,0,0,30.6958,C,First,man,True,A,Cherbourg,no,True
535,1,2,female,7.0,0,2,26.2500,S,Second,child,False,NaN,Southampton,yes,False


In [117]:
df.shape

(891, 15)

#Quick Checklist of things to do with the dataset:

####0. Overview
####1. Missing values summary
####2. Duplicates
####3. Data Type validation
####4. Constant and quasi constant columns
####5. ID like columns
####6. String inconsistencies
####7. High null columns
####8. High zero columns

###Adding Passenger_id

In [118]:
df["Passenger_Id"] = df.index + 1

#MOVE PASSENGER ID COL TO START OF DF
cols = list(df.columns)
cols = [cols[-1]] + cols[:-1]
df = df[cols]

In [119]:
df.head(2)

,Passenger_Id,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,1,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,2,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False


In [120]:
#move survived col to the end
column_to_move = df.pop("survived")
df.insert(15, "survived", column_to_move)
df.head()

,Passenger_Id,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,survived
0,1,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,0
1,2,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,1
2,3,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,1
3,4,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,1
4,5,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,0


Remove any constant columns and ids from the dataset

when training the model or EDA, because it doesn't

AFFECT THE PREDICTION BECAUSE IT IS NOT INFLUENCING ANY FEATURES OR TARGET

#0. Overview

In [121]:
df.shape

(891, 16)

In [122]:
df.describe()

,Passenger_Id,pclass,age,sibsp,parch,fare,survived
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000,891.000000
mean,446.000000,2.308642,29.699118,0.523008,0.381594,32.204208,0.383838
std,257.353842,0.836071,14.526497,1.102743,0.806057,49.693429,0.486592
min,1.000000,1.000000,0.420000,0.000000,0.000000,0.000000,0.000000
25%,223.500000,2.000000,20.125000,0.000000,0.000000,7.910400,0.000000
50%,446.000000,3.000000,28.000000,0.000000,0.000000,14.454200,0.000000
75%,668.500000,3.000000,38.000000,1.000000,0.000000,31.000000,1.000000
max,891.000000,3.000000,80.000000,8.000000,6.000000,512.329200,1.000000


In [123]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   Passenger_Id  891 non-null    int64   
 1   pclass        891 non-null    int64   
 2   sex           891 non-null    object  
 3   age           714 non-null    float64 
 4   sibsp         891 non-null    int64   
 5   parch         891 non-null    int64   
 6   fare          891 non-null    float64 
 7   embarked      889 non-null    object  
 8   class         891 non-null    category
 9   who           891 non-null    object  
 10  adult_male    891 non-null    bool    
 11  deck          203 non-null    category
 12  embark_town   889 non-null    object  
 13  alive         891 non-null    object  
 14  alone         891 non-null    bool    
 15  survived      891 non-null    int64   
dtypes: bool(2), category(2), float64(2), int64(5), object(5)
memory usage: 87.6+ KB


In [124]:
df.dtypes

,0
Passenger_Id,int64
pclass,int64
sex,object
age,float64
sibsp,int64
parch,int64
fare,float64
embarked,object
class,category
who,object


In [125]:
df.columns

Index(['Passenger_Id', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone', 'survived'],
      dtype='object')

In [126]:
#tells about categorical or not values
df.nunique()

,0
Passenger_Id,891
pclass,3
sex,2
age,88
sibsp,7
parch,7
fare,248
embarked,3
class,3
who,3


#1. Missing values summary

In [127]:
df.isna().sum()

,0
Passenger_Id,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [128]:
# age -- 177 -- impute w median or mean
# embarked -- 2 same here
# deck -- 688 -- probably will have to drop this column after checking correlations
# embark_town	2 -- could drop the 2 rows

In [129]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_percentage = (df.isna().mean()*100).sort_values(ascending=False)

missing_df = pd.DataFrame({"missing_count":missing_count,"missing_percentage":missing_percentage})
missing_df[missing_df["missing_count"]>0]

,missing_count,missing_percentage
deck,688,77.216611
age,177,19.865320
embark_town,2,0.224467
embarked,2,0.224467


#2.Duplicates

In [130]:
df.duplicated().groupby(df.duplicated()).count()

,0
False,891


In [131]:
df.duplicated().sum()

np.int64(0)

#3. Data Type validation

Incase we gotta change the data type:

In [132]:
df["survived"] = df["survived"].astype("int")

#4. Constant and Quasi Constant Columns
--to find which columns to remove because they are constant and unaffecting to the target or features

In [133]:
n_rows=len(df)
nunique = df.nunique()
constant_cols = nunique[nunique==1].index.to_list()
print(constant_cols)

[]


In [134]:
df.nunique().sort_values(ascending=True)

,0
sex,2
alone,2
survived,2
adult_male,2
alive,2
pclass,3
embarked,3
class,3
embark_town,3
who,3


###To find quasi constant columns
quasi constant means that are mostly constant, they will not provide sufficient information to the model either.
in this case if like 90% of the values are the same, they are not useful?ig

In [135]:
quasi_constant_cols = []
for col in df.columns:
  top_freq= df[col].value_counts(normalize=True,dropna=False).values[0]
  if top_freq > 0.9 and col not in constant_cols:
    quasi_constant_cols.append(col)

quasi_constant_cols

[]

In [136]:
for col in df.columns:
  print (df[col].value_counts(normalize=True)*100)

Passenger_Id
891    0.112233
1      0.112233
2      0.112233
3      0.112233
4      0.112233
         ...   
16     0.112233
15     0.112233
14     0.112233
13     0.112233
12     0.112233
Name: proportion, Length: 891, dtype: float64
pclass
3    55.106622
1    24.242424
2    20.650954
Name: proportion, dtype: float64
sex
male      64.758698
female    35.241302
Name: proportion, dtype: float64
age
24.00    4.201681
22.00    3.781513
18.00    3.641457
28.00    3.501401
30.00    3.501401
           ...   
24.50    0.140056
0.67     0.140056
0.42     0.140056
34.50    0.140056
74.00    0.140056
Name: proportion, Length: 88, dtype: float64
sibsp
0    68.237935
1    23.456790
2     3.142536
4     2.020202
3     1.795735
8     0.785634
5     0.561167
Name: proportion, dtype: float64
parch
0    76.094276
1    13.243547
2     8.978676
5     0.561167
3     0.561167
4     0.448934
6     0.112233
Name: proportion, dtype: float64
fare
8.0500     4.826038
13.0000    4.713805
7.8958     4.264871
7.7

In [137]:
#df.groupby("sex")["survived"].mean()

#5. ID like columns

columns where number of unique values is closer to number of df rows
pretty much checking if a col is constant another way

In [138]:
n_rows = len(df)
id_like_cols = []
for col in df.columns:
  if df[col].nunique(dropna=False) > 0.95*n_rows:
    id_like_cols.append(col)

id_like_cols

['Passenger_Id']

#6. String Inconsistencies

Check if string data have any inconsistencies

In [146]:
object_cols = df.select_dtypes(include=['object', "category"]).columns.to_list()
object_cols

['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive']

In [147]:
for col in object_cols:
  print(df[col].unique())

['male' 'female']
['S' 'C' 'Q' nan]
['Third', 'First', 'Second']
Categories (3, object): ['First', 'Second', 'Third']
['man' 'woman' 'child']
[NaN, 'C', 'E', 'G', 'D', 'A', 'B', 'F']
Categories (7, object): ['A', 'B', 'C', 'D', 'E', 'F', 'G']
['Southampton' 'Cherbourg' 'Queenstown' nan]
['no' 'yes']


####stripping and lowercase

In [157]:
for cols in object_cols:
  df[cols] = df[cols].str.strip()
  df[cols] = df[cols].str.lower()
  df[cols]=df[cols].replace("NaN",np.nan)

In [158]:
df[object_cols]

,sex,embarked,class,who,deck,embark_town,alive
0,male,s,third,man,NaN,southampton,no
1,female,c,first,woman,c,cherbourg,yes
2,female,s,third,woman,NaN,southampton,yes
3,female,s,first,woman,c,southampton,yes
4,male,s,third,man,NaN,southampton,no
...,...,...,...,...,...,...,...
886,male,s,second,man,NaN,southampton,no
887,female,s,first,woman,b,southampton,yes
888,female,s,third,woman,NaN,southampton,no
889,male,c,first,man,c,cherbourg,yes


In [159]:
for col in object_cols:
  print(df[col].unique())

['male' 'female']
['s' 'c' 'q' nan]
['third' 'first' 'second']
['man' 'woman' 'child']
[nan 'c' 'e' 'g' 'd' 'a' 'b' 'f']
['southampton' 'cherbourg' 'queenstown' nan]
['no' 'yes']


#7. High null columns


In [176]:
high_null_threshold = 40
null_values = df.isna().sum().sort_values(ascending=False)
null_percentage = (df.isna().mean()*100).sort_values(ascending=False)

null_df = pd.DataFrame({"null_count":null_values,"null_percentage":null_percentage})
null_df[null_df["null_percentage"]>high_null_threshold]

,null_count,null_percentage
deck,688,77.216611


#High Zero Columns

In [180]:
df.dtypes

,0
Passenger_Id,int64
pclass,int64
sex,object
age,float64
sibsp,int64
parch,int64
fare,float64
embarked,object
class,object
who,object


In [194]:
numeric_columns = df.select_dtypes(include=np.number).columns.to_list()
#numeric_columns.remove("survived")
numeric_columns

['Passenger_Id', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'survived']

In [219]:
high_zero_threshold = 40
zero_values = (df[numeric_columns]==0).mean()*100
zero_values.sort_values(ascending=False)

,0
parch,76.094276
sibsp,68.237935
survived,61.616162
fare,1.683502
age,0.000000
Passenger_Id,0.000000
pclass,0.000000


In [225]:
zero_values[zero_values>high_zero_threshold]

,0
sibsp,68.237935
parch,76.094276
survived,61.616162
